# cross-product-normal — worked example 2: Quad normal averaged from two triangle fans

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `cross-product-normal`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A planar quadrilateral `(A, B, C, D)` can be split into two triangles `(A,B,C)` and `(A,C,D)` that share the diagonal `A-C`. Each triangle contributes a normal via the cross product, and averaging the two unit normals gives a robust face normal that is less sensitive to a single near-degenerate triangle.

## Worked solution

**Step 1 — split the quad.** We fan from vertex `A`: triangle 1 is `(A, B, C)`, triangle 2 is `(A, C, D)`. They share edge `A-C`, so together they tile the quad with no overlap.

**Step 2 — normal of triangle 1.** Edges `e1 = B - A`, `e2 = C - A`; `n1 = cross(e1, e2)`. Using the same anchor `A` and the same winding order (B then C) keeps the orientation consistent.

**Step 3 — normal of triangle 2.** Edges `f1 = C - A`, `f2 = D - A`; `n2 = cross(f1, f2)`. Again anchored at `A` with order C then D, matching the counter-clockwise winding `A->B->C->D`.

**Step 4 — normalize each.** We divide each cross product by its norm so neither triangle dominates the average just because it is larger in area.

**Step 5 — average and renormalize.** Summing two unit vectors and dividing by the sum's norm yields the unit bisector direction. For a perfectly planar quad both normals are identical, so the average is exactly that normal.

In [ ]:
def quad_normal(A: Tensor, B: Tensor, C: Tensor, D: Tensor) -> Tensor:
    n1 = t.linalg.cross(B - A, C - A)
    n2 = t.linalg.cross(C - A, D - A)
    u1 = n1 / n1.norm()
    u2 = n2 / n2.norm()
    s = u1 + u2
    return s / s.norm()

A = t.tensor([0.0, 0.0, 0.0])
B = t.tensor([1.0, 0.0, 0.0])
C = t.tensor([1.0, 1.0, 0.0])
D = t.tensor([0.0, 1.0, 0.0])
print(quad_normal(A, B, C, D))